In [1]:
import warnings
warnings.filterwarnings( 'ignore' )
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer

In [2]:
dir = '../../../data/top30groups/'

df = pd.read_csv(f'{dir}df_top30_300.csv')

split = int(len(df) * 0.7)

traindata = df.iloc[:split]
testdata = df.iloc[split:]


print(f'shape {traindata}', traindata.shape)
print(f'shape {testdata}', testdata.shape)

shape       Unnamed: 0  iyear  imonth  iday  extended  country  region  provstate  \
0          77017   2016      10    15         0      160       5          0   
1          82749   2017      10    14         1      160       5          1   
2          50575   2011      12     3         0      160       5          1   
3          79039   2017       2    19         1      160       5          2   
4          75463   2016       7    14         0      160       5          1   
...          ...    ...     ...   ...       ...      ...     ...        ...   
6295       29578   1992       1    30         0       45       3        363   
6296       78947   2017       2    14         0       45       3        384   
6297       14203   1985       7    10         0       45       3        385   
6298       36611   1997      10    22         0       45       3        366   
6299       20491   1988      10    11         0       45       3        363   

      city  latitude  ...  targtype1  target1

In [3]:
def fit_factorize(df, text_features):
    mappings = {}
    for col in text_features:
        df[col], uniques = pd.factorize(df[col])
        mappings[col] = uniques
    return df, mappings

def apply_factorize(df, mappings):
    for col, uniques in mappings.items():
        df[col] = df[col].apply(lambda x: uniques.get_loc(x) if x in uniques else -1)
    return df


In [4]:
text_features = traindata.select_dtypes(include='object').columns
text_features = text_features.drop('gname')

trainp1, gname_mappings = fit_factorize(traindata, text_features)
testp1 = apply_factorize(testdata, gname_mappings)

In [5]:
def split_data(dftrain, dftest):
    Ytrain = dftrain['gname']
    Xtrain = dftrain.drop(columns=['gname'])
    Ytest = dftest['gname']
    Xtest = dftest.drop(columns=['gname'])
    return Xtrain, Ytrain, Xtest, Ytest

def find_best_rfc(Xtrain, Ytrain):

     params = {
          'criterion': ["gini", "entropy"],
          'n_estimators': [5, 10, 20, 50, 100, 150, 200, 300, 500],
          'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
          'max_features': ['sqrt', 'log2']
          }

     rfc = RandomForestClassifier(random_state=42)

     tscv = TimeSeriesSplit(n_splits=5)


     grid_search =GridSearchCV(estimator=rfc, param_grid=params, cv = tscv)

     grid_search.fit(Xtrain, Ytrain)
     best_rfc = grid_search.best_estimator_
     best_rfc_params = grid_search.best_params_
     #print(best_dt)
     return best_rfc_params

In [ ]:
best_rfc_params = []
Xtrains = []
Ytrains = []
truths = []
Xtests = []

Xtrain, Ytrain, Xtest, Ytest = split_data(traindata, testdata)
print(f'Finding best rfc for parition')
best_rfc_param = find_best_rfc(Xtrain, Ytrain)
print('---------------------------------')
best_rfc_params.append(best_rfc_param)
Xtrains.append(Xtrain)
Ytrains.append(Ytrain)
truths.append(Ytest)
Xtests.append(Xtest)

Finding best rfc for parition


In [ ]:
accuracies = []

for i in range(len(Xtests)):
    print(f'partition {i+1}:')
    model = RandomForestClassifier(**best_rfc_params[i], random_state=42)
    model.fit(Xtrains[i], Ytrains[i])
    y_pred_rfc = model.predict(Xtests[i])
    accuracy_rfc = accuracy_score(truths[i], y_pred_rfc)
    accuracies.append(accuracy_rfc)
    print(f"Accuracy: {accuracy_rfc * 100:.2f}%")
    print('-------------------------------------------------')

In [ ]:
print("Unique labels in Ytest:", sorted(set(Ytest)))
print("Unique labels in y_pred_rfc:", sorted(set(y_pred_rfc)))


In [ ]:
Ytest.value_counts()

In [ ]:
from matplotlib.colors import LogNorm


all_classes = ['Al-Shabaab', 'Basque Fatherland and Freedom (ETA)', 'Boko Haram',
               'Farabundo Marti National Liberation Front (FMLN)',
               'Irish Republican Army (IRA)',
               'Liberation Tigers of Tamil Eelam (LTTE)',
               'National Liberation Army of Colombia (ELN)',
               "New People's Army (NPA)", 'Palestinians',
               'Revolutionary Armed Forces of Colombia (FARC)',
               'Shining Path (SL)', 'Taliban']

cm = confusion_matrix(Ytest, y_pred_rfc, labels=all_classes)

df_cm = pd.DataFrame(cm, index=all_classes, columns=all_classes)
print(df_cm)
# Optional: Add class names

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='.0f', cmap='Blues',
            xticklabels=all_classes, yticklabels=all_classes,
            norm=LogNorm())  # 👈 this makes smaller values visible
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
print("Ytest unique:", sorted(set(Ytest)))
print("y_pred_rfc unique:", sorted(set(y_pred_rfc)))
print("Labels used in cm:", all_classes)


In [ ]:
Ytest.value_counts()

In [ ]:
print(classification_report(Ytest, y_pred_rfc))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Get importances
importances = model.feature_importances_
feature_names = Xtrain.columns

# Create a dataframe
feat_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Plot top features
feat_importance_df.head(20).plot(kind='barh', x='Feature', y='Importance', figsize=(8, 6))
plt.title("Top 20 Feature Importances (Random Forest)")
plt.show()
